<a href="https://colab.research.google.com/github/Heng1222/MalSem-Decon/blob/main/method2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. 安裝必要套件
!pip install -U sentence-transformers jieba pandas numpy zipfile

import pandas as pd
import numpy as np
import jieba
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==========================================
# 第一部分：配置與模組化函數
# ==========================================
def read_timeline_xlsx(path):
    # 直接解析 xlsx 的 XML，避免缺少 openpyxl 時讀檔失敗。
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rows = []

    def cell_text(cell):
        inline = cell.find("a:is", ns)
        value = cell.find("a:v", ns)
        return "".join(inline.itertext()) if inline is not None else (value.text if value is not None else "")

    with zipfile.ZipFile(path) as zf:
        root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        sheet_rows = root.find("a:sheetData", ns)
        header_map = {}

        for row_idx, row in enumerate(sheet_rows):
            row_values = {}
            for cell in row:
                ref = cell.attrib.get("r", "")
                col = "".join(ch for ch in ref if ch.isalpha())
                row_values[col] = cell_text(cell)

            if row_idx == 0:
                header_map = row_values
                continue

            rows.append({header_map.get(col, col): value for col, value in row_values.items()})

    return pd.DataFrame(rows)

def get_embedding_model(model_name='paraphrase-multilingual-MiniLM-L12-v2'):
    """
    初始化 Embedding 模型。
    參考 method1 的多國語言配置。
    """
    print(f"正在載入模型: {model_name}...")
    return SentenceTransformer(model_name)

def clean_text(text):
    """清理文字，僅保留中文字符"""
    return re.sub(r'[^\u4e00-\u9fa5]', '', str(text))

def get_word_frequencies(corpus):
    """對語料庫進行斷詞並統計詞頻"""
    words = []
    for text in corpus:
        words.extend(jieba.lcut(clean_text(text)))
    return Counter([w for w in words if len(w) > 1]) # 排除單字

def calculate_log_odds_ratio(target_counts, ref_counts):
    """
    實作 Log-odds Ratio 演算法。
    用於找出在目標桶中相對於基準桶更顯著的特徵詞。
    """
    all_words = set(list(target_counts.keys()) + list(ref_counts.keys()))
    n_target = sum(target_counts.values())
    n_ref = sum(ref_counts.values())

    log_odds = {}
    for word in all_words:
        # 加上平滑項 (Prior) 避免除以零
        p_i = (target_counts[word] + 0.1) / (n_target + 0.1)
        q_i = (ref_counts[word] + 0.1) / (n_ref + 0.1)
        # Log-odds 核心公式
        log_odds[word] = np.log(p_i / (1 - p_i)) - np.log(q_i / (1 - q_i))

    return sorted(log_odds.items(), key=lambda x: x[1], reverse=True)

def validate_by_projection(word, current_bad_center, model, threshold=0.5):
    """
    向量投影/相似度校驗。
    確保新詞在語意空間中與目前的惡意概念中心足夠接近。
    """
    word_vec = model.encode([word])
    sim = cosine_similarity(word_vec, current_bad_center.reshape(1, -1))[0][0]
    return sim >= threshold, sim

# ==========================================
# 第二部分：主流程實作
# ==========================================

def run_bootstrapping_pipeline(df_corpus, initial_seeds, n_target=20):
    """
    執行完整的關鍵詞擴增迭代
    """
    model = get_embedding_model()
    target_keywords = set(initial_seeds)
    all_texts = df_corpus.copy()
    # 確保所有資料都是字串，並排除空值 (NaN)
    all_texts = df_corpus.copy().dropna().astype(str).reset_index(drop=True)

    # 過濾掉空白字串 (例如只包含空格的貼文)
    all_texts = all_texts[all_texts.str.strip() != ""].reset_index(drop=True)

    # 預先計算全語料 Embedding 以加速搜尋
    print("正在計算全語料 Embedding...")
    all_embeddings = model.encode(all_texts.tolist(), show_progress_bar=True)
    print(all_embeddings.shape)
    iteration = 0
    while len(target_keywords) < n_target:
        iteration += 1
        print(f"\n>>> 迭代 {iteration} | 目前關鍵詞數: {len(target_keywords)}")

        # 步驟 1 & 2: 建立壞桶與好桶 (基準桶)
        pattern = '|'.join([re.escape(k) for k in target_keywords])
        bad_mask = all_texts.str.contains(pattern, na=False)
        bad_bucket = all_texts[bad_mask]
        good_bucket = all_texts[~bad_mask] # 修正後的「好桶」定義為一般背景語料

        if bad_bucket.empty:
            print("警告：壞桶為空，請檢查初始種子。")
            break

        # 步驟 3: 語意擴張 (透過 BERT 找出相似內容形成大壞桶)
        bad_vecs = model.encode(bad_bucket.tolist())
        bad_center = bad_vecs.mean(axis=0)

        # 尋找與壞桶中心最相似的前 50 篇文章
        sims = cosine_similarity(bad_center.reshape(1, -1), all_embeddings)[0]
        big_bad_indices = sims.argsort()[-50:][::-1]
        big_bad_bucket = all_texts.iloc[big_bad_indices]

        # 步驟 4: 計算 Log-odds 找出特徵詞
        target_counts = get_word_frequencies(big_bad_bucket)
        ref_counts = get_word_frequencies(good_bucket.sample(min(2000, len(good_bucket))))

        candidate_words = calculate_log_odds_ratio(target_counts, ref_counts)

        # 步驟 5: 校驗並更新關鍵詞集
        added_count = 0
        for word, score in candidate_words:
            if word not in target_keywords:
                # 向量投影校驗：防止語意偏移 (Semantic Drift)
                is_valid, sim_score = validate_by_projection(word, bad_center, model)
                if is_valid:
                    target_keywords.add(word)
                    added_count += 1
                    print(f"  [新增] {word} (Log-odds: {score:.2f}, Sim: {sim_score:.2f})")

            if len(target_keywords) >= n_target or added_count >= 5:
                break

        if added_count == 0:
            print("無法再找到符合條件的新詞，停止擴增。")
            break

    return list(target_keywords)

# ==========================================
# 第三部分：資料載入與執行
# ==========================================

# 讀取你的 Series 資料 (此處範例假設讀取 CSV 後轉 Series)
# fb_data = pd.read_csv("your_data.csv")['content']
posts = read_timeline_xlsx("time_line.xlsx")
text_columns = ["content", "share_content", "attachment_title", "attachment_description"]
posts[text_columns] = posts[text_columns].fillna("").astype(str)
posts["text"] = posts[text_columns].agg(" ".join, axis=1).map(clean_text)
fb_data = posts["text"]
# 模擬測試數
# test_data = pd.Series([
#     "博弈娛樂城，儲值就送點數", "專業下注教學，百家樂必勝", "今日早餐吃蛋餅好滿足",
#     "投資詐騙手法大公開", "這家珊瑚色唇膏好漂亮", "運動彩券即時比分",
#     "地下錢莊高利貸追債", "線上麻將現金交易", "台北美食餐廳推薦"
# ])
# 負面種子詞：來自既有的 negative keyword 清單。
seeds = (
    pd.read_csv("negative_keywords.txt", header=None, encoding="utf-8")[0]
    .dropna()
    .astype(str)
    .tolist()
)
# seeds = ["下注", "賭博", "娛樂城"]
final_keywords = run_bootstrapping_pipeline(fb_data, seeds, n_target=50)

# 儲存結果
output_df = pd.DataFrame(final_keywords, columns=["keyword"])
output_df.to_csv("output.csv", index=False, encoding="utf-8-sig")
print("\n任務完成！結果已儲存至 output.csv")

ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement zipfile (from versions: none)
ERROR: No matching distribution found for zipfile
正在載入模型: paraphrase-multilingual-MiniLM-L12-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

正在計算全語料 Embedding...


Batches:   0%|          | 0/1129 [00:00<?, ?it/s]

(36109, 384)

>>> 迭代 1 | 目前關鍵詞數: 30


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.720 seconds.
DEBUG:jieba:Loading model cost 0.720 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


  [新增] 老爹 (Log-odds: 6.86, Sim: 0.52)
  [新增] 實車 (Log-odds: 6.74, Sim: 0.59)
  [新增] 店主 (Log-odds: 6.46, Sim: 0.53)
  [新增] 瓜子 (Log-odds: 6.28, Sim: 0.52)
  [新增] 千葉 (Log-odds: 6.28, Sim: 0.55)

>>> 迭代 2 | 目前關鍵詞數: 35
  [新增] 包子 (Log-odds: 5.98, Sim: 0.58)
  [新增] 燦坤幣 (Log-odds: 5.98, Sim: 0.63)
  [新增] 旅行箱 (Log-odds: 5.98, Sim: 0.54)
  [新增] 滿元 (Log-odds: 5.98, Sim: 0.56)
  [新增] 滿享 (Log-odds: 5.98, Sim: 0.60)

>>> 迭代 3 | 目前關鍵詞數: 40
  [新增] 滿額 (Log-odds: 7.41, Sim: 0.53)
  [新增] 集賢 (Log-odds: 7.14, Sim: 0.57)
  [新增] 燦坤 (Log-odds: 7.08, Sim: 0.58)
  [新增] 燦坤台 (Log-odds: 7.01, Sim: 0.65)
  [新增] 特典 (Log-odds: 6.67, Sim: 0.54)

>>> 迭代 4 | 目前關鍵詞數: 45
  [新增] 溫並 (Log-odds: 6.47, Sim: 0.54)
  [新增] 首家 (Log-odds: 6.47, Sim: 0.53)
  [新增] 洽門市 (Log-odds: 6.31, Sim: 0.61)
  [新增] 入手 (Log-odds: 6.31, Sim: 0.56)
  [新增] 價限時 (Log-odds: 6.31, Sim: 0.53)

任務完成！結果已儲存至 output.csv


In [6]:
kw = pd.read_csv("output.csv")
print(kw)

   keyword
0     強力過件
1       槓桿
2     不問經歷
3      快易貸
4       全額
5      價限時
6       日領
7       集賢
8       燦坤
9      免頭期
10      老爹
11      致富
12     燦坤台
13      清算
14     洽門市
15     招待所
16     燦坤幣
17      千葉
18      八大
19      店主
20     旅行箱
21      酒店
22      信貸
23     娛樂城
24      包子
25      炫富
26      瓜子
27      實車
28      入手
29      首家
30      運彩
31      破產
32     企業貸
33      夜場
34      溫並
35      債務
36      借款
37      法拍
38      貸款
39      滿元
40      精品
41      卡債
42      厭世
43      滿額
44     自救會
45      博弈
46      特典
47      滿享
48      二胎
49      協商
